In [115]:
print("Hello World")

Hello World


In [116]:
from openai import OpenAI
import re

from dotenv import load_dotenv
_ = load_dotenv()

client = OpenAI()

In [117]:
class Agent:

    """ 
    A Class representing an AI agent that can interact with the OpenAI API.
    """
    
    def __init__(self, system=""):
        """
            Initialize the agent with a system prompt.

            Args:
                system (str): The system prompt to guide the agent's behavior.
        """

        self.system = system
        self.messages = []
        if self.system:
            # If a system prompt is provided, add it to the message history
            self.messages.append({"role": "system", "content": system})

        
    def __call__(self, messages):
            """
            Allow agent to be called directly with a message.

            Args:
                message (str): The userinput message.

            Returns:
                str: The agent's response to the input message.
            """

            #Add user message to conversation history
            self.messages.append({"role": "user", "content": messages})
            #Execute the conversation and get response
            result = self.execute()
            #Add agent's response to conversation history
            self.messages.append({"role": "assistant", "content": result})
            return result
        

    def execute(self):
            """
            Execute the conversation by sending the entire conversation history to the OpenAI API.

            Returns:
                str: The content of the agent's response.
            """

            response = client.chat.completions.create(
                model="gpt-4o",
                temperature=0,
                messages=self.messages
            )
            return response.choices[0].messages.content

In [118]:
# Define prompt for the agent

prompt = """
You run in a loop of Thought, Action, Pause, Observation.
At the end of the loop you must output your final answer
Use Thought to describe what you are thinking
Use Action to describe what you want to do
Use Pause to pause and wait for the observation
Use Observation to describe what you observed after pausing


Your available actions are:
calculate_total_price:
e.g. calculate_total_price: appl2 2, banana 3
Runs a price calculation for the given items and quantities.

get_fruit_price:
e.g. get_fruit_price: apple
returns the price of the given fruit.

Example session:
Question: What is the total price if I buy 2 apples and 3 bananas?
Thought: I should calculate the price by getting the price of each fruit first and summing them up.
Action: get_fruit_price: apple
PAUSE

Observation: The price of an apple is $1.5.

Action: get_fruit_price: banana
PAUSE

Observation: The price of a banana is $1.2

Action: calculate_total_price: apple 2, banana 3
PAUSE

You then output:

Answer: The total price for 2 apples and 3 bananas is $6.6.

""".strip()

In [119]:
#Price lookup for fruits
fruit_prices = {
    "apple": 1.5,
    "banana": 1.2,
    "orange": 1.3,
    "grape": 2.0,
    "mango": 2.5
}

#Function to calculate the price of a specific fruit
def get_fruit_price(fruit):
    if fruit in fruit_prices:
        return f"The price of a {fruit} is ${fruit_prices[fruit]}."
    else:
        return f"Sorry, I don't have the price for {fruit}."
    
#Function to calculate the total price for given items and quantities
def calculate_total_price(items_quantities):
    total = 0.0
    fruits_list = fruits.split(",")
    for item in fruits_list:
        fruit, quantity = item.split(", ")
        quantity = int(quantity)
        if fruit in fruit_prices:
            total += fruit_prices[fruit] * quantity
        else:
            return f"Sorry, I don't have the price for {fruit}."
    return f"The total price is ${total:.2f}."

#Mapping of action names to functions
known_actions = {
    "get_fruit_price": get_fruit_price,
    "calculate_total_price": calculate_total_price 
}

In [120]:
#Run a query through the agent
action_re = re.compile(r'^Action: (\w+): (.*)$')

def query(question):
    bot = Agent(prompt)
    result = bot(question) #__call__
    print(result)
    actions = [
        action_re.match(a)
        for a in result.split('\n')
        if action_re.match(a)
    ]
    if actions:
        action, action_input = actions[0].groups()
        if action not in known_actions:
            raise Exception(f"Unknown action: {action}: {action_input}")
        print (f"  -- running {action} {action_input}")
        observation = known_actions[action](action_input)
        print("Observation:", observation)
    else:
        return

In [121]:
import os
api_key = os.environ.get("OPENAI_API_KEY")
print(f"API Key loaded: {api_key[:10] if api_key else 'NOT FOUND'}")

API Key loaded: sk-proj-hb


In [122]:
import socket
try:
    socket.create_connection(("api.openai.com", 443), timeout=5)
    print("Connection successful")
except Exception as e:
    print(f"Blocked or error: {e}")

Connection successful


In [123]:
# query("What is the total price of an orange?")

import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")

if api_key:
    print(f"✓ API Key found")
    print(f"  First 10 chars: {api_key[:10]}")
    print(f"  Last 10 chars: {api_key[-10:]}")
    print(f"  Length: {len(api_key)}")
    print(f"  Starts with 'sk_': {api_key.startswith('sk_')}")
else:
    print("✗ API Key NOT found in environment")

✓ API Key found
  First 10 chars: sk-proj-hb
  Last 10 chars: 5PnQPK_RwA
  Length: 164
  Starts with 'sk_': False


In [125]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# Test message list (not self.messages)
test_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"}
]

response = client.chat.completions.create(
    model="gpt-4o",
    temperature=0,
    messages=test_messages
)

print(response.choices[0].message.content)

APIConnectionError: Connection error.